In [33]:
import pandas as pd
import numpy as np
import re
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix
from joblib import Parallel, delayed
import implicit

# 1. Data Loading and Preprocessing
- start by loading the playlist and track data, then build an interaction matrix where each row represents a playlist and each column represents a unique track. A cell is set to 1 if the track is in the playlist.

In [34]:
# Load playlists and tracks CSV files
playlists = pd.read_csv('/Users/xavierhua/Documents/GitHub/spotifynd/phase5_model_development/playlist_with_segment.csv')
tracks = pd.read_csv('/Users/xavierhua/Documents/GitHub/spotifynd/phase5_model_development/tracks_new.csv')

# Reset index so rows run from 0 to n-1
# Filter playlists by cluster and reset the index to ensure contiguous row indices
playlists = playlists[playlists['cluster'] == 1].reset_index(drop=True)

# Function to parse a string into a list of integers
def parse_track_list(s):
    return [int(x) for x in re.findall(r'\d+', s)]

# Parse track list columns
playlists['track_idx_list'] = playlists['track_idx_list'].apply(parse_track_list)
playlists['tracks_to_predict'] = playlists['tracks_to_predict'].apply(parse_track_list)

# (Optionally use a subset—for example, first 1000 playlists)
# playlists = playlists[:1000]

# Build mapping: playlist id -> list of tracks (training set)
playlist_ids = playlists['playlist_idx'].tolist()
playlist_tracks = dict(zip(playlist_ids, playlists['track_idx_list']))

# Create a sorted list of all unique track IDs in the training playlists
unique_tracks = set()
for tlist in playlist_tracks.values():
    unique_tracks.update(tlist)
unique_tracks = sorted(list(unique_tracks))

# Map each track ID to a column index in the interaction matrix
track_to_col = {track: idx for idx, track in enumerate(unique_tracks)}

# Build the dense interaction matrix: rows = playlists, columns = tracks
num_playlists = len(playlist_ids)
num_tracks = len(unique_tracks)
interaction_matrix_dense = np.zeros((num_playlists, num_tracks), dtype=np.int32)
for i, pid in enumerate(playlist_ids):
    for track in playlist_tracks[pid]:
        j = track_to_col[track]
        interaction_matrix_dense[i, j] = 1

print("Filtered playlists:", num_playlists)
print("Unique tracks:", num_tracks)
print("Dense interaction matrix shape:", interaction_matrix_dense.shape)

# Convert dense matrix to sparse (for ALS, KNN, etc.)
sparse_interaction = csr_matrix(interaction_matrix_dense)

# Build the test set from 'tracks_to_predict'
test_items = {}
for i, row in playlists.iterrows():
    test_items[i] = [track_to_col[t] for t in row['tracks_to_predict'] if t in track_to_col]

Filtered playlists: 6141
Unique tracks: 90761
Dense interaction matrix shape: (6141, 90761)


# 2. Define Models and Prediction Functions

### 2.1 ITEM-BASED CF USING CO-OCCURRENCE 
- For item-based CF, we compute similarities between tracks based on the playlists in which they appear. For a given track, find similar tracks that tend to co-occur in playlists, then recommend those.
- Computes an item co-occurrence matrix from the dense playlist–track matrix.
- For a given playlist, sums co-occurrence counts of items that co-occur with the playlist’s items.
- Filters out items already in the playlist by setting their scores to -∞.
- Provides a wrapper function to return the recommendation scores for that playlist.

In [35]:
# Compute the co-occurrence matrix using the sparse interaction matrix.
cooc_matrix_sparse = (sparse_interaction.T).dot(sparse_interaction)

def predict_item_based_cooccurrence_sparse(playlist_idx, dense_matrix, cooc_sparse):
    """
    For a given playlist, sum the co-occurrence counts for items that co-occur with
    items already in the playlist, using the sparse co-occurrence matrix.
    """
    in_playlist = np.where(dense_matrix[playlist_idx] > 0)[0]
    if len(in_playlist) == 0:
        return np.zeros(dense_matrix.shape[1])
    
    # Sum the relevant rows from the sparse co-occurrence matrix.
    scores = np.array(cooc_sparse[in_playlist, :].sum(axis=0), dtype=np.float64).flatten()
    scores[in_playlist] = -np.inf
    return scores

def predict_item_based_wrapper_sparse(playlist_idx, dense_matrix):
    return predict_item_based_cooccurrence_sparse(playlist_idx, dense_matrix, cooc_matrix_sparse)

### 2.2 User-Based CF
- For user-based CF, we compute similarities between playlists. For a given playlist, find other similar playlists and recommend tracks that are present in these similar playlists but missing in the current one.
- Computes a full user–user cosine similarity matrix from the training data.
- For a given playlist, aggregates weighted track scores from similar playlists.
- Excludes tracks already in the target playlist by setting their scores to -∞.
- Provides a wrapper that computes similarity and returns prediction scores.

In [36]:
# --- USER-BASED CF (using full cosine similarity) ---
def compute_user_similarity(train_matrix):
    return cosine_similarity(train_matrix)

def predict_user_based(playlist_idx, train_matrix, user_sim_matrix):
    sim_scores = user_sim_matrix[playlist_idx]
    predicted_scores = sim_scores.dot(train_matrix)
    # Exclude tracks already present
    existing_indices = np.where(train_matrix[playlist_idx].flatten() > 0)[0]
    predicted_scores[existing_indices] = -np.inf
    return predicted_scores

# Precompute user similarity once
user_sim_matrix = cosine_similarity(interaction_matrix_dense)

def predict_user_based_wrapper(playlist_idx, dense_matrix):
    # Use the precomputed similarity matrix
    return predict_user_based(playlist_idx, dense_matrix, user_sim_matrix)

### 2.3 SVD Matrix Factorization
- We use truncated SVD to factorize the interaction matrix into latent factors. Then, recommendations are generated by computing the dot product of the latent factors.
- Performs truncated SVD on the dense interaction matrix to obtain latent factors.
- Rescales latent factors with the square root of singular values.
- Computes recommendation scores via the dot product of playlist and track latent factors.
- Filters out tracks already in the playlist (sets their scores to -∞).
- Provides a wrapper function for convenience.

In [37]:
# --- MF via SVD ---
latent_dim = 100
svd = TruncatedSVD(n_components=latent_dim, random_state=42)
U = svd.fit_transform(interaction_matrix_dense)      # (num_playlists x latent_dim)
Sigma = svd.singular_values_                         # (latent_dim,)
VT = svd.components_                                 # (latent_dim x num_tracks)
sqrt_sigma = np.sqrt(Sigma)
P_svd = U * sqrt_sigma                               # Playlist latent factors
Q_svd = (VT.T * sqrt_sigma)                          # Track latent factors

def predict_mf(playlist_idx, train_matrix, P, Q):
    predicted_scores = P[playlist_idx].dot(Q.T)
    existing_indices = np.where(train_matrix[playlist_idx].flatten() > 0)[0]
    predicted_scores[existing_indices] = -np.inf
    return predicted_scores

def predict_mf_wrapper(playlist_idx, dense_matrix):
    return predict_mf(playlist_idx, dense_matrix, P_svd, Q_svd)

### 2.4 ALS
- Use ALS on a sparse implicit feedback matrix to learn latent factors for playlists and tracks, then generate recommendations by computing the dot product of these factors.
- Sets ALS hyperparameters and scales the sparse interaction matrix.
- Trains an ALS model (using implicit) to learn user and item latent factors.
- Computes scores via the dot product of the learned latent factors.
- Excludes items already in the playlist by setting their scores to -∞.
- Provides a wrapper function for prediction convenience.

In [38]:
# --- ALS Matrix Factorization ---
als_latent_dim = 100
regularization = 0.1
iterations = 20
alpha_val = 40  # Confidence scaling factor

als_model = implicit.als.AlternatingLeastSquares(factors=als_latent_dim,
                                                 regularization=regularization,
                                                 iterations=iterations,
                                                 random_state=42)
data_conf = (sparse_interaction * alpha_val).astype('double')
als_model.fit(data_conf)
P_als = als_model.user_factors   # (num_playlists x latent_dim)
Q_als = als_model.item_factors   # (num_tracks x latent_dim)

def predict_als(playlist_idx, train_matrix, P_als, Q_als):
    predicted_scores = P_als[playlist_idx].dot(Q_als.T)
    existing_indices = np.where(train_matrix[playlist_idx].flatten() > 0)[0]
    predicted_scores[existing_indices] = -np.inf
    return predicted_scores

def predict_als_wrapper(playlist_idx, dense_matrix):
    return predict_als(playlist_idx, dense_matrix, P_als, Q_als)

  0%|          | 0/20 [00:00<?, ?it/s]

# 3. Evaluation Functions
- **Ranking Metrics:**
  - **Hit Ratio @K:** Fraction of playlists with at least one test track in the top K.
  - **MRR:** Average reciprocal rank of the first relevant test track.
  - **MAP @K:** Mean average precision over the top K recommendations.

- **Confusion Matrix & F2 Score:**
  - Compute TP, FP, FN, TN for top K recommendations.
  - Derive per–playlist precision, recall, and F2 (β = 2), and aggregate these metrics (both average and overall).

- **BPR–Style AUC:**
  - For each test track, sample negatives (from non–interacted items) and compute the fraction where the test track scores higher.
  - Average these fractions across playlists to yield an overall ranking AUC.

### 3.1 Ranking Metrics
- **compute_metrics_for_playlist(predicted_scores, test_indices, k=10):**
  - Ranks items by predicted scores and selects top-k.
  - Computes:
    - **Hit:** 1 if any test item is in top-k, else 0.
    - **MRR:** Reciprocal rank of the first relevant item.
    - **AP:** Average precision over top-k.
- **evaluate_model(dense_matrix, test_items, predict_func, k=10):**
  - Iterates over playlists, gets predictions using the provided function.
  - Aggregates Hit, MRR, and AP across all playlists.
  - Returns average Hit Ratio, MRR, and MAP over the test set.

In [39]:
def compute_metrics_for_playlist(predicted_scores, test_indices, k=10):
    ranked_indices = np.argsort(-predicted_scores)
    top_k = ranked_indices[:k]
    hit = 1 if any(t in top_k for t in test_indices) else 0

    precisions = []
    num_hits = 0
    mrr = 0.0
    for rank_idx, track_idx in enumerate(ranked_indices[:k]):
        if track_idx in test_indices:
            num_hits += 1
            precisions.append(num_hits / (rank_idx + 1))
            if mrr == 0.0:
                mrr = 1.0 / (rank_idx + 1)
    ap = np.mean(precisions) if precisions else 0.0
    return hit, mrr, ap

def evaluate_model(dense_matrix, test_items, predict_func, k=10):
    hit_total = 0
    mrr_total = 0
    ap_total = 0
    num_playlists = len(test_items)
    
    for playlist_idx, test_indices in test_items.items():
        predicted_scores = predict_func(playlist_idx, dense_matrix)
        hit, mrr, ap = compute_metrics_for_playlist(predicted_scores, test_indices, k)
        hit_total += hit
        mrr_total += mrr
        ap_total += ap
    
    hit_ratio = hit_total / num_playlists
    mrr_avg = mrr_total / num_playlists
    map_avg = ap_total / num_playlists
    return hit_ratio, mrr_avg, map_avg

### Confusion Matrix and F2 Evaluation (Top-K Only)
- **compute_confusion_for_playlist_topk:**
  - Considers only the top K recommended items.
  - Computes:
    - **TP:** Number of top-K items that are in the test set.
    - **FP:** K minus TP.
    - **FN:** Test items missing from the top K.
  - Calculates precision (TP/K), recall (TP/(TP+FN)), and F2 score (with β=2).
  
- **evaluate_confusion_matrix_topk:**
  - Iterates over all playlists and applies the above function.
  - Aggregates per–playlist metrics to compute average and overall precision, recall, and F2.

In [40]:
# --- Confusion Matrix and F2 Evaluation (Top-K Only) ---
def compute_confusion_for_playlist_topk(predicted_scores, test_indices, k):
    top_k = set(np.argsort(-predicted_scores)[:k])
    test_set = set(test_indices)
    TP = len(top_k & test_set)
    FP = k - TP
    FN = len(test_set - top_k)
    precision = TP / k if k > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    beta = 2
    if (beta**2 * precision + recall) > 0:
        f2 = (1 + beta**2) * precision * recall / (beta**2 * precision + recall)
    else:
        f2 = 0
    return TP, FP, FN, precision, recall, f2

def evaluate_confusion_matrix_topk(dense_matrix, test_items, predict_func, k=10):
    total_TP, total_FP, total_FN = 0, 0, 0
    total_precision, total_recall, total_f2 = 0, 0, 0
    count = 0
    
    for playlist_idx, test_indices in test_items.items():
        predicted_scores = predict_func(playlist_idx, dense_matrix)
        TP, FP, FN, precision, recall, f2 = compute_confusion_for_playlist_topk(predicted_scores, test_indices, k)
        total_TP += TP
        total_FP += FP
        total_FN += FN
        total_precision += precision
        total_recall += recall
        total_f2 += f2
        count += 1
    
    avg_precision = total_precision / count if count > 0 else 0
    avg_recall = total_recall / count if count > 0 else 0
    avg_f2 = total_f2 / count if count > 0 else 0
    overall_precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
    overall_recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
    beta = 2
    if (beta**2 * overall_precision + overall_recall) > 0:
        overall_f2 = (1 + beta**2) * overall_precision * overall_recall / (beta**2 * overall_precision + overall_recall)
    else:
        overall_f2 = 0
    
    return (total_TP, total_FP, total_FN), (avg_precision, avg_recall, avg_f2), (overall_precision, overall_recall, overall_f2)

### 3.3 BPR AUC
- **evaluate_bpr_auc:**
  - Iterates over each playlist in the test set.
  - For each playlist:
    - Computes predicted scores.
    - Identifies negative items (not in the training set).
    - For each positive (test) item, samples negatives (up to num_samples).
    - Compares the positive score against sampled negatives to compute a per–item AUC.
  - Aggregates per–item AUCs across playlists and returns the average AUC.

In [41]:
# Optionally, you can also implement a BPR–style ranking AUC evaluation
def evaluate_bpr_auc(dense_matrix, test_items, predict_func, num_samples=100):
    auc_total = 0
    count_total = 0
    num_playlists = len(test_items)
    for playlist_idx, test_indices in test_items.items():
        predicted_scores = predict_func(playlist_idx, dense_matrix)
        train_indices = set(np.where(dense_matrix[playlist_idx] > 0)[0])
        negatives = np.array([i for i in range(dense_matrix.shape[1]) if i not in train_indices])
        if len(negatives) == 0:
            continue
        for pos in test_indices:
            if len(negatives) < num_samples:
                sampled_negs = negatives
            else:
                sampled_negs = np.random.choice(negatives, num_samples, replace=False)
            pos_score = predicted_scores[pos]
            correct = sum(pos_score > predicted_scores[neg] for neg in sampled_negs)
            auc_total += correct / len(sampled_negs)
            count_total += 1
    return auc_total / count_total if count_total > 0 else 0

# 4. Evaluate All Models

In [42]:
K_eval = 50
print("Evaluating Models with top-K =", K_eval)

Evaluating Models with top-K = 50


### 4.1 Evaluating Item-Based CF

In [43]:
hit_item, mrr_item, map_item = evaluate_model(interaction_matrix_dense, test_items, predict_item_based_wrapper_sparse, k=K_eval)
(conf_item, avg_conf_item, overall_conf_item) = evaluate_confusion_matrix_topk(interaction_matrix_dense, test_items, predict_item_based_wrapper_sparse, k=K_eval)
auc_item = evaluate_bpr_auc(interaction_matrix_dense, test_items, predict_item_based_wrapper_sparse)
print(f"Item-based CF (KNN on items) - Hit Ratio: {hit_item:.3f}, MRR: {mrr_item:.3f}, MAP: {map_item:.3f}, Total: {hit_item+mrr_item+map_item:.3f}")
print(f"Confusion Matrix Totals: {conf_item}")
print(f"Avg Precision, Recall, F2: {avg_conf_item}")
print(f"Overall Precision, Recall, F2: {overall_conf_item}")
print(f"BPR–style AUC: {auc_item:.3f}\n")

Item-based CF (KNN on items) - Hit Ratio: 0.546, MRR: 0.113, MAP: 0.085, Total: 0.744
Confusion Matrix Totals: (6921, 300129, 48829)
Avg Precision, Recall, F2: (0.02254030288226697, 0.11854714474401909, 0.06382651823236768)
Overall Precision, Recall, F2: (0.022540302882266734, 0.12414349775784754, 0.06528629374587303)
BPR–style AUC: 0.924



### 4.2 Evaluating User-Based CF

In [44]:
hit_user, mrr_user, map_user = evaluate_model(interaction_matrix_dense, test_items, predict_user_based_wrapper, k=K_eval)
(conf_user, avg_conf_user, overall_conf_user) = evaluate_confusion_matrix_topk(interaction_matrix_dense, test_items, predict_user_based_wrapper, k=K_eval)
auc_user = evaluate_bpr_auc(interaction_matrix_dense, test_items, predict_user_based_wrapper)
print(f"User-based CF - Hit Ratio: {hit_user:.3f}, MRR: {mrr_user:.3f}, MAP: {map_user:.3f}, Total: {hit_user+mrr_user+map_user:.3f}")
print(f"Confusion Matrix Totals: {conf_user}")
print(f"Avg Precision, Recall, F2: {avg_conf_user}")
print(f"Overall Precision, Recall, F2: {overall_conf_user}")
print(f"BPR–style AUC: {auc_user:.3f}\n")

KeyboardInterrupt: 

### 4.3 SVD

In [45]:
hit_mf, mrr_mf, map_mf = evaluate_model(interaction_matrix_dense, test_items, predict_mf_wrapper, k=K_eval)
(conf_mf, avg_conf_mf, overall_conf_mf) = evaluate_confusion_matrix_topk(interaction_matrix_dense, test_items, predict_mf_wrapper, k=K_eval)
auc_mf = evaluate_bpr_auc(interaction_matrix_dense, test_items, predict_mf_wrapper)
print(f"MF-SVD - Hit Ratio: {hit_mf:.3f}, MRR: {mrr_mf:.3f}, MAP: {map_mf:.3f}, Total: {hit_mf+mrr_mf+map_mf:.3f}")
print(f"Confusion Matrix Totals: {conf_mf}")
print(f"Avg Precision, Recall, F2: {avg_conf_mf}")
print(f"Overall Precision, Recall, F2: {overall_conf_mf}")
print(f"BPR–style AUC: {auc_mf:.3f}\n")

MF-SVD - Hit Ratio: 0.624, MRR: 0.139, MAP: 0.102, Total: 0.865
Confusion Matrix Totals: (8374, 298676, 47376)
Avg Precision, Recall, F2: (0.02727243120013089, 0.14343302755613535, 0.07723328760515298)
Overall Precision, Recall, F2: (0.027272431200130273, 0.15020627802690584, 0.07899254787284218)
BPR–style AUC: 0.926



### 4.4 ALS

In [46]:
hit_als, mrr_als, map_als = evaluate_model(interaction_matrix_dense, test_items, predict_als_wrapper, k=K_eval)
(conf_als, avg_conf_als, overall_conf_als) = evaluate_confusion_matrix_topk(interaction_matrix_dense, test_items, predict_als_wrapper, k=K_eval)
auc_als = evaluate_bpr_auc(interaction_matrix_dense, test_items, predict_als_wrapper)
print(f"ALS - Hit Ratio: {hit_als:.3f}, MRR: {mrr_als:.3f}, MAP: {map_als:.3f}, Total: {hit_als+mrr_als+map_als:.3f}")
print(f"Confusion Matrix Totals: {conf_als}")
print(f"Avg Precision, Recall, F2: {avg_conf_als}")
print(f"Overall Precision, Recall, F2: {overall_conf_als}")
print(f"BPR–style AUC: {auc_als:.3f}")

ALS - Hit Ratio: 0.601, MRR: 0.117, MAP: 0.088, Total: 0.805
Confusion Matrix Totals: (7913, 299137, 47837)
Avg Precision, Recall, F2: (0.025771047060739814, 0.13623110862974167, 0.07309615497402314)
Overall Precision, Recall, F2: (0.025771047060739294, 0.1419372197309417, 0.07464390151872465)
BPR–style AUC: 0.887
